# 6.0 Dataframe maestro

Este notebook tiene como objetivo **construir el dataframe maestro** (`corpus_maestro.parquet`), que une todos los metadatos del scraping, la asignación temática de BERTopic y las etiquetas ideológicas del modelo ganador (`LoRA QKV r8`) en un único fichero.


**Fuentes que se fusionan:**

| Fichero | Aportación |
|---|---|
| `corpus_limpio_filtrado.parquet` | titular, abstract, noticia, autor, fecha, url, tema original |
| `{medio}_limpio_filtrado.json` × 6 | orden de IDs para alinear con el array de topics |
| `bertopic_nooutlier_topics.npy` | topic_id (0-28, -1 = sin clasificar) por artículo |
| `cluster_labels.json` | topic_id → nombre_topic, macrotema |
| `corpus_clasificado_final.csv` | ideologia, confianza, p_izquierda/centro/derecha, origen |

## 0. Configuración e imports

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

# ── Rutas ──────────────────────────────────────────────────────────────────
DATA_DIR    = Path('../data/processed')
OUTPUT_DIR  = Path('../output')
CLUSTER_DIR = Path('../output/clustering_6medios')

MEDIOS = ['elconfidencial', 'eldiario', 'elmundo', 'elpais', 'okdiario', 'publico']


# Paleta de colores
COLORS_IDEO = {
    'IZQUIERDA': '#e15759',
    'CENTRO':    '#f28e2b',
    'DERECHA':   '#4e79a7',
}

print('Configuración lista')

Configuración lista


## 1. Construcción del dataframe maestro

El merge se hace en tres pasos:

```
parquet (241 240)  ──┐
                     ├── LEFT JOIN por id ──→ df (241 240)
topics  (240 903)  ──┘
                     |
                     └── LEFT JOIN por id ──→ df_maestro
CSV     (241 240)  ──┘
```

**Topics tiene 337 artículos menos que el parquet,** porque durante la generación de embeddings (notebook `3_embedings.ipynb`), 337 artículos fallaron por OOM y se guardaron como vectores de ceros. El notebook `4_clustering.ipynb` los detecta y los elimina de los ficheros JSON y de los `.npy` de embeddings antes de ejecutar BERTopic, pero **no modifica el parquet**. La clasificación ideológica sí se hizo sobre el parquet completo, así que esos 337 artículos tienen `label_ideologia` pero nunca entraron al clustering y no tienen `macrotema`.

Como el reranking opera por macrotema y estos artículos nunca podrían aparecer en ningún resultado, **se eliminan del dataframe maestro** tras el merge.


### 1.1 Metadatos del scraping

Cargamos el parquet limpio y derivamos campos temporales. Eliminamos `texto` (versión procesada para embeddings), `len_texto` y `truncado` (no hay ningún artículo truncado en el corpus filtrado).

In [2]:
df_meta = pd.read_parquet(DATA_DIR / 'corpus_limpio_filtrado.parquet')

# Parsear fecha y derivar año / mes
df_meta['fecha_dt'] = pd.to_datetime(df_meta['fecha'], errors='coerce', format='mixed')
df_meta['año']      = df_meta['fecha_dt'].dt.year
df_meta['mes']      = df_meta['fecha_dt'].dt.strftime('%Y-%m')  # string 'YYYY-MM'

# Eliminar columnas de preprocesamiento interno que no aportan al análisis
df_meta = df_meta.drop(columns=['texto', 'len_texto', 'truncado'], errors='ignore')

print(f'Parquet cargado: {df_meta.shape[0]:,} artículos × {df_meta.shape[1]} columnas')
print(f'Rango de fechas: {df_meta["fecha_dt"].min().date()} → {df_meta["fecha_dt"].max().date()}')
print(f'Medios: {df_meta["medio"].value_counts().to_dict()}')

Parquet cargado: 241,240 artículos × 13 columnas
Rango de fechas: 2023-03-14 → 2026-04-09
Medios: {'El País': 64936, 'OKDiario': 52023, 'elDiario.es': 38494, 'El Confidencial': 36718, 'Público': 26057, 'El Mundo': 23012}


### 1.2 Asignación de topics (BERTopic)

El array `bertopic_nooutlier_topics.npy` tiene **240 903 entradas**, una por artículo, en el mismo orden en que el notebook 4 cargó los JSONs (orden `MEDIOS`). Reconstruimos ese orden para hacer un join correcto por `id`.

In [3]:
# Reconstruir la lista de IDs en el mismo orden que usó BERTopic
ids_ordered = []
for medio in MEDIOS:
    path_json = DATA_DIR / f'{medio}_limpio_filtrado.json'
    noticias  = json.loads(path_json.read_text(encoding='utf-8'))
    ids_ordered.extend([n['id'] for n in noticias])

topics = np.load(CLUSTER_DIR / 'bertopic_nooutlier_topics.npy', allow_pickle=True)

assert len(ids_ordered) == len(topics), (
    f'Desajuste: {len(ids_ordered):,} IDs vs {len(topics):,} topics'
)

df_topics = pd.DataFrame({'id': ids_ordered, 'topic_id': topics.astype(int)})
print(f'IDs con topic asignado: {len(df_topics):,}')
print(f'Sin clasificar (topic_id = -1): {(df_topics["topic_id"] == -1).sum():,}')

IDs con topic asignado: 240,903
Sin clasificar (topic_id = -1): 21


In [4]:
# Cargar el mapping topic_id → nombre_topic, macrotema
with open(CLUSTER_DIR / 'cluster_labels.json', encoding='utf-8') as f:
    raw_labels = json.load(f)

CLUSTER_LABELS = {int(k): v for k, v in raw_labels.items()}

df_topics['nombre_topic'] = df_topics['topic_id'].map(
    lambda t: CLUSTER_LABELS[t]['nombre']    if t in CLUSTER_LABELS else 'sin_clasificar'
)
df_topics['macrotema'] = df_topics['topic_id'].map(
    lambda t: CLUSTER_LABELS[t]['macrotema'] if t in CLUSTER_LABELS else 'sin_clasificar'
)

print('Topics asignados:')
print(df_topics['macrotema'].value_counts().to_string())

Topics asignados:
macrotema
politica_nacional      70771
sucesos_seguridad      37862
economia_social        36030
internacional          25813
justicia_corrupcion    25454
derechos_sociales      21624
macroeconomia          20858
tecnologia              2470
sin_clasificar            21


### 1.3 Etiquetas ideológicas

Cargamos solo las columnas necesarias de `corpus_clasificado_final.csv` y renombramos:
- `label` → `label_ideologia`
- `label_source` → `ideologia_origen`
- `confianza` → `ideologia_confianza`

In [5]:
df_labels = pd.read_csv(
    OUTPUT_DIR / 'corpus_clasificado_final.csv',
    usecols=['id', 'label', 'confianza', 'p_izquierda', 'p_centro', 'p_derecha', 'label_source']
)

df_labels = df_labels.rename(columns={
    'label':        'label_ideologia',
    'label_source': 'ideologia_origen',
    'confianza':     'ideologia_confianza',
})

print(f'Etiquetas cargadas: {len(df_labels):,}')
print('\nDistribución por origen:')
print(df_labels['ideologia_origen'].value_counts().rename('artículos').to_string())
print('\nDistribución por ideología:')
print(df_labels['label_ideologia'].value_counts().rename('artículos').to_string())

Etiquetas cargadas: 241,240

Distribución por origen:
ideologia_origen
inference    238862
train          1902
test            238
val             238

Distribución por ideología:
label_ideologia
CENTRO       100831
IZQUIERDA     83449
DERECHA       56960


### 1.4 Merge completo

Unimos las tres fuentes con LEFT JOIN sobre `id`. Así los 337 artículos sin topic obtienen `macrotema = 'sin_clasificar'` en lugar de desaparecer.

In [6]:
# Paso 1: metadatos + topics
df = df_meta.merge(
    df_topics[['id', 'topic_id', 'nombre_topic', 'macrotema']],
    on='id', how='left'
)

# Rellenar NaN de los artículos sin topic (zero-embedding)
df['topic_id']    = df['topic_id'].fillna(-1).astype(int)
df['nombre_topic'] = df['nombre_topic'].fillna('sin_clasificar')
df['macrotema']   = df['macrotema'].fillna('sin_clasificar')

# Paso 2: añadir etiquetas ideológicas
df = df.merge(df_labels, on='id', how='left')

# Ordenar columnas según el diseño del dataframe maestro
cols_orden = [
    'id', 'medio', 'url', 'titular', 'autor', 'autores_lista',
    'fecha', 'fecha_dt', 'año', 'mes',
    'abstract', 'noticia', 'tema',
    'topic_id', 'nombre_topic', 'macrotema',
    'label_ideologia', 'ideologia_confianza', 'p_izquierda', 'p_centro', 'p_derecha', 'ideologia_origen'
]
df = df[cols_orden]

print(f'Dataframe maestro: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Columnas: {list(df.columns)}')
# Eliminar los 337 artículos con embedding cero que fueron excluidos del clustering
# Tienen label_ideologia pero no macrotema → nunca aparecerían en el reranking
n_antes = len(df)
df = df[df['topic_id'] != -1].reset_index(drop=True)
print(f'Eliminados {n_antes - len(df):,} artículos sin topic → {len(df):,} artículos en el dataframe maestro')


Dataframe maestro: 241,240 filas × 22 columnas
Columnas: ['id', 'medio', 'url', 'titular', 'autor', 'autores_lista', 'fecha', 'fecha_dt', 'año', 'mes', 'abstract', 'noticia', 'tema', 'topic_id', 'nombre_topic', 'macrotema', 'label_ideologia', 'ideologia_confianza', 'p_izquierda', 'p_centro', 'p_derecha', 'ideologia_origen']
Eliminados 358 artículos sin topic → 240,882 artículos en el dataframe maestro


### 1.5 Verificación del dataframe maestro

In [7]:
print('Distribución por macrotema')
print(df['macrotema'].value_counts().rename('artículos').to_string())

print('\n=== Distribución por ideología ===')
print(df['label_ideologia'].value_counts().rename('artículos').to_string())

print('\n=== Distribución por origen ===')
print(df['ideologia_origen'].value_counts().rename('artículos').to_string())

print('\n=== NaN por columna clave ===')
print(df[['titular', 'fecha_dt', 'macrotema', 'label_ideologia', 'ideologia_confianza']].isna().sum().to_string())

Distribución por macrotema
macrotema
politica_nacional      70771
sucesos_seguridad      37862
economia_social        36030
internacional          25813
justicia_corrupcion    25454
derechos_sociales      21624
macroeconomia          20858
tecnologia              2470

=== Distribución por ideología ===
label_ideologia
CENTRO       100705
IZQUIERDA     83319
DERECHA       56858

=== Distribución por origen ===
ideologia_origen
inference    238504
train          1902
test            238
val             238

=== NaN por columna clave ===
titular                   0
fecha_dt                  6
macrotema                 0
label_ideologia           0
ideologia_confianza    2378


In [8]:
# Muestra de 3 artículos aleatorios
df.sample(3, random_state=42)[
    ['id', 'medio', 'titular', 'macrotema', 'nombre_topic', 'label_ideologia', 'ideologia_confianza', 'ideologia_origen']
].T

,21243,181070,64710
id,elpais_21312,okdiario_19783,elpais_68365
medio,El País,OKDiario,El País
titular,El Govern de ERC blinda al comisario jefe de los Mossos a dos días de las el...,Los socialistas europeos se sienten engañados por Sánchez: «¿De qué sirvió p...,El hombre que alza la voz en el vagón del metro
macrotema,justicia_corrupcion,politica_nacional,internacional
nombre_topic,corrupcion_judicial,politica_partidos,opinion_internacional
label_ideologia,IZQUIERDA,DERECHA,IZQUIERDA
ideologia_confianza,0.8701,1.0,1.0
ideologia_origen,inference,inference,inference


### 1.6 Guardado

Guardamos el dataframe maestro en formato Parquet (compresión snappy). Incluye la columna `noticia` con el texto completo, por lo que el fichero puede pesar varios GB.

In [9]:
MAESTRO_PATH = Path('../data/final/corpus_maestro.parquet')
MAESTRO_PATH.parent.mkdir(exist_ok=True)
df.to_parquet(MAESTRO_PATH, index=False, compression='snappy')

size_mb = MAESTRO_PATH.stat().st_size / 1024 / 1024
print(f'Guardado en {MAESTRO_PATH}')
print(f'Tamaño: {size_mb:.1f} MB')

Guardado en ../data/final/corpus_maestro.parquet
Tamaño: 638.7 MB
